# House price dataset preparation
- Use this notebook to generate dataset.
- Date: 9-3-2026

In [1]:
import numpy as np
import pandas as pd

In [6]:
from sklearn.datasets import fetch_openml

data = fetch_openml(data_id=42165, as_frame=True, parser="auto").frame

for var in data.select_dtypes(include="object").columns:
    data[var] = data[var].str.strip("'")

# drop id
data.drop(columns=["Id"], inplace=True)

# cast as object
data['MSSubClass'] = data['MSSubClass'].astype('str')

# rows and columns of the data
print(data.shape)

# visualise the dataset
data.head()

(1460, 80)


,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## Impute categorical variables

In [20]:
# impute categorical variables

cat_vars = [var for var in data.columns if data[var].dtype == 'object']

data[cat_vars] = data[cat_vars].fillna('Missing')

## Impute numerical variables

In [21]:
# impute numerical variables

import numpy as np

# Identify genuinely numerical variables with missing values.
# This correctly filters for numeric dtypes (like int, float).
num_vars = data.select_dtypes(include=np.number).columns[
    data.select_dtypes(include=np.number).isnull().any()
].tolist()

# Ensure 'SalePrice' is not in num_vars if it's numeric and has nulls,
# as it's the target variable and should not be imputed with feature means.
if 'SalePrice' in num_vars:
    num_vars.remove('SalePrice')

# Calculate means only for the identified numerical variables
mean_dict = data[num_vars].mean().to_dict()

data = data.fillna(mean_dict)

## Capture time since

In [23]:
# capture time since

for var in ['YearBuilt', 'YearRemodAdd', 'GarageYrBlt']:
    if 'YrSold' in data.columns and var in data.columns:
        data[var] = data['YrSold'] - data[var]

if 'YrSold' in data.columns:
    data.drop(['YrSold'], axis=1, inplace=True)

In [25]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Binarize skewed variables

In [11]:
# binarize skewed variables

skewed = [
    'BsmtFinSF2', 'LowQualFinSF', 'EnclosedPorch',
    '3SsnPorch', 'ScreenPorch', 'MiscVal'
]

for var in skewed:
    data[var] = np.where(data[var]==0, 0, 1)

## Re-map strings to numbers

In [12]:
# re-map strings to numbers, which determine quality

qual_mappings = {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5, 'Missing': 0, 'NA': 0}

qual_vars = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond',
             'HeatingQC', 'KitchenQual', 'FireplaceQu',
             'GarageQual', 'GarageCond',
            ]

for var in qual_vars:
    data[var] = data[var].map(qual_mappings)

In [13]:
exposure_mappings = {'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4, "Missing": 0}

var = 'BsmtExposure'

data[var] = data[var].map(exposure_mappings)

In [14]:
finish_mappings = {'Missing': 0, 'NA': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6}

finish_vars = ['BsmtFinType1', 'BsmtFinType2']

for var in finish_vars:
    data[var] = data[var].map(finish_mappings)

In [15]:
garage_mappings = {'Missing': 0, 'NA': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3}

var = 'GarageFinish'

data[var] = data[var].map(garage_mappings)

In [16]:
fence_mappings = {'Missing': 0, 'NA': 0, 'MnWw': 1, 'GdWo': 2, 'MnPrv': 3, 'GdPrv': 4}

var = 'Fence'

data[var] = data[var].map(fence_mappings)

## Capture all quality variables

In [17]:
# capture all quality variables

qual_vars  = qual_vars + finish_vars + ['BsmtExposure','GarageFinish','Fence']

# capture the remaining categorical variables
# (those that we did not re-map)

cat_others = [
    var for var in cat_vars if var not in qual_vars
]

len(cat_others)

0

## Remove rare categories

In [18]:
# remove rare categories

def find_frequent_labels(df, var, rare_perc):

    # function finds the labels that are shared by more than
    # a certain % of the houses in the dataset

    df = df.copy()

    tmp = df.groupby(var)[var].count() / len(df)

    return tmp[tmp > rare_perc].index


for var in cat_others:

    # find the frequent categories
    frequent_ls = find_frequent_labels(data, var, 0.01)

    # replace rare categories by the string "Rare"
    data[var] = np.where(data[var].isin(
        frequent_ls), data[var], 'Rare')

## One Hot Encoding

In [24]:
# one hot encoding

if len(cat_others) > 0:
    data = pd.concat([
        data,
        pd.get_dummies(data[cat_others], drop_first=True).astype(int)],
        axis=1,
    )
    data.drop(columns=cat_others, inplace=True)
else:
    print("No additional categorical variables to encode.")

data.head()

No additional categorical variables to encode.


,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,SaleType,SaleCondition,SalePrice
0,60,RL,65.0,8450,Pave,Missing,Reg,Lvl,AllPub,Inside,...,0,0,Missing,2.939502,Missing,0,2,WD,Normal,208500
1,20,RL,80.0,9600,Pave,Missing,Reg,Lvl,AllPub,FR2,...,0,0,Missing,2.939502,Missing,0,5,WD,Normal,181500
2,60,RL,68.0,11250,Pave,Missing,IR1,Lvl,AllPub,Inside,...,0,0,Missing,2.939502,Missing,0,9,WD,Normal,223500
3,70,RL,60.0,9550,Pave,Missing,IR1,Lvl,AllPub,Corner,...,0,0,Missing,2.939502,Missing,0,2,WD,Abnorml,140000
4,60,RL,84.0,14260,Pave,Missing,IR1,Lvl,AllPub,FR2,...,0,0,Missing,2.939502,Missing,0,12,WD,Normal,250000


In [26]:
# check absence of na in the data set
[var for var in data.columns if data[var].isnull().sum() > 0]

[]

In [29]:
data.to_csv("/content/drive/MyDrive/Colab Notebooks/Machine Learning Explainability/houseprice_prep.csv", index=False)

In [31]:
## load generated csv
dataset = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Machine Learning Explainability/houseprice_prep.csv")
dataset.head()

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,SaleType,SaleCondition,SalePrice
0,60,RL,65.0,8450,Pave,Missing,Reg,Lvl,AllPub,Inside,...,0,0,Missing,2.939502,Missing,0,2,WD,Normal,208500
1,20,RL,80.0,9600,Pave,Missing,Reg,Lvl,AllPub,FR2,...,0,0,Missing,2.939502,Missing,0,5,WD,Normal,181500
2,60,RL,68.0,11250,Pave,Missing,IR1,Lvl,AllPub,Inside,...,0,0,Missing,2.939502,Missing,0,9,WD,Normal,223500
3,70,RL,60.0,9550,Pave,Missing,IR1,Lvl,AllPub,Corner,...,0,0,Missing,2.939502,Missing,0,2,WD,Abnorml,140000
4,60,RL,84.0,14260,Pave,Missing,IR1,Lvl,AllPub,FR2,...,0,0,Missing,2.939502,Missing,0,12,WD,Normal,250000


That concludes the feature engineering section.

# Additional Resources

- [Feature Engineering for Machine Learning](https://www.trainindata.com/p/feature-engineering-for-machine-learning) - Online Course
- [Packt Feature Engineering Cookbook](https://www.amazon.com/Python-Feature-Engineering-Cookbook-transforming-dp-1804611301/dp/1804611301) - Book